<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [31]:
import io
import os

import torch
import torch.nn as nn
import timm
import numpy as np
import pandas as pd
import albumentations as A

import torchmetrics
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [32]:
# Датасет публичный, поэтому HF_TOKEN для его загрузки не обязателен.
# Для приватных репозиториев задайте HF_TOKEN во внешнем окружении.

In [33]:
splits = {
    "train": "data/train-00000-of-00001-0d6632d599c29801.parquet",
    "validation": "data/validation-00000-of-00001-c6b77a557eeedd52.parquet",
    "test": "data/test-00000-of-00001-866d29d8989ea915.parquet",
}

dataset_root = "hf://datasets/Francesco/halo-infinite-angel-videogame/"
df_train = pd.read_parquet(dataset_root + splits["train"])
df_validation = pd.read_parquet(dataset_root + splits["validation"])
df_test = pd.read_parquet(dataset_root + splits["test"])

Создаем датасет для предобработки данных

In [34]:
df_train.head(3)

,image_id,image,width,height,objects
0,311,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [573, 574], 'area': [1748, 15756], 'bbo..."
1,67,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [137, 138], 'area': [14136, 88392], 'bb..."
2,161,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,640,640,"{'id': [301, 302], 'area': [3016, 33768], 'bbo..."


In [35]:
def _flatten_values(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        values = []
        for item in value:
            values.extend(_flatten_values(item))
        return values
    return [value]


def _extract_boxes(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        if len(value) == 4 and all(np.isscalar(item) for item in value):
            return [[float(item) for item in value]]
        boxes = []
        for item in value:
            boxes.extend(_extract_boxes(item))
        return boxes
    return []


class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe["objects"])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe["image"])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = np.array(Image.open(io.BytesIO(row["bytes"])).convert("RGB"))

        labels = [
            int(label) - 1
            for label in _flatten_values(row["category"])
            if label is not None
        ]
        boxes = _extract_boxes(row["bbox"])
        if len(boxes) != len(labels):
            raise ValueError(
                f"Image {row['image_id']} has {len(boxes)} boxes and "
                f"{len(labels)} labels"
            )

        boxes = np.asarray(boxes, dtype=np.float32)
        if boxes.size:
            boxes[:, 2:] += boxes[:, :2]
        boxes = boxes.tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image = transformed["image"]
            boxes = transformed["bboxes"]
            labels = transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target = {
            "image_id": row["image_id"],
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }
        return image, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images), targets

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:

In [73]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
image_size = 320

bbox_params = A.BboxParams(
    format="pascal_voc",
    label_fields=["labels"],
    min_visibility=0.3,
)

train_transform = A.Compose(
    [
        A.Resize(image_size, image_size),
        A.RandomResizedCrop(size=(image_size, image_size), scale=(0.7, 1.0), p=0.5),
        A.HorizontalFlip(p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2,
                p=1.0,
            ),
            A.HueSaturationValue(
                hue_shift_limit=20,
                sat_shift_limit=30,
                val_shift_limit=20,
                p=1.0,
            ),
        ], p=0.7),
        A.OneOf([
            A.GaussNoise(p=1.0),
            A.CoarseDropout(
                num_holes_range=(1, 8),
                hole_height_range=(0.02, 0.12),
                hole_width_range=(0.02, 0.12),
                fill=0,
                p=1.0,
            ),
        ], p=0.3),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=bbox_params,
)

test_transform = A.Compose(
    [
        A.Resize(image_size, image_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=bbox_params,
)

Не забываем инициализировать наш датасет

In [74]:
train_dataset = HaloDataset(df_train, transform=train_transform)
validation_dataset = HaloDataset(df_validation, transform=test_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [77]:
class Backbone(nn.Module):
    def __init__(self, model_name="efficientnet_b0", out_indices=(2, 3, 4), unfreeze_last=0):
        super().__init__()
        if unfreeze_last < 0:
            raise ValueError("unfreeze_last must be non-negative")

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True,
            out_indices=out_indices,
        )
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        blocks = list(self.backbone.children())
        if unfreeze_last > len(blocks):
            raise ValueError(
                f"unfreeze_last={unfreeze_last} exceeds the number of blocks "
                f"({len(blocks)})"
            )
        for block in blocks[-unfreeze_last:] if unfreeze_last else []:
            for parameter in block.parameters():
                parameter.requires_grad = True

    def forward(self, x):
        return self.backbone(x)

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [78]:
import torch.nn as nn
import torch.nn.functional as F


class Neck(nn.Module):
    """Feature Pyramid Network для нескольких выходов backbone."""

    def __init__(self, in_channels, out_channels=256):
        super().__init__()
        if len(in_channels) == 0:
            raise ValueError("in_channels must contain at least one value")

        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(channels, out_channels, kernel_size=1)
            for channels in in_channels
        ])
        self.output_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
            for _ in in_channels
        ])

    def forward(self, features):
        if len(features) != len(self.lateral_convs):
            raise ValueError(
                f"Expected {len(self.lateral_convs)} feature maps, "
                f"got {len(features)}"
            )

        pyramid = [
            lateral(feature)
            for lateral, feature in zip(self.lateral_convs, features)
        ]
        for level in range(len(pyramid) - 2, -1, -1):
            pyramid[level] = pyramid[level] + F.interpolate(
                pyramid[level + 1],
                size=pyramid[level].shape[-2:],
                mode="nearest",
            )

        return [
            output_conv(feature)
            for output_conv, feature in zip(self.output_convs, pyramid)
        ]

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [79]:
class Head(nn.Module):
    """Decoupled head для одного уровня FPN."""

    def __init__(self, in_channels, num_anchors, num_classes, hidden_channels=None):
        super().__init__()
        if num_anchors <= 0 or num_classes <= 0:
            raise ValueError("num_anchors and num_classes must be positive")

        hidden_channels = in_channels if hidden_channels is None else hidden_channels
        self.num_anchors = num_anchors
        self.num_classes = num_classes

        self.cls_tower = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
        )
        self.reg_tower = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
        )
        self.cls_head = nn.Conv2d(
            hidden_channels, num_anchors * num_classes, kernel_size=1
        )
        self.reg_head = nn.Conv2d(
            hidden_channels, num_anchors * 5, kernel_size=1
        )

    def forward(self, x):
        cls_logits = self.cls_head(self.cls_tower(x))
        bbox_preds = self.reg_head(self.reg_tower(x))
        return cls_logits, bbox_preds

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [80]:
from torchvision.models.detection.anchor_utils import AnchorGenerator


class Detector(nn.Module):
    """Detector assembled from the backbone, FPN neck, and one head per level."""

    def __init__(
        self,
        backbone_model_name="efficientnet_b0",
        neck_n_channels=128,
        num_classes=4,
        anchor_sizes=(32, 64, 128),
        anchor_ratios=(0.5, 1.0, 2.0),
        input_size=(320, 320),
        unfreeze_last=0,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.backbone = Backbone(
            backbone_model_name,
            out_indices=(2, 3, 4),
            unfreeze_last=unfreeze_last,
        )
        in_channels = self.backbone.backbone.feature_info.channels()
        reductions = self.backbone.backbone.feature_info.reduction()
        num_levels = len(in_channels)

        if len(anchor_sizes) != num_levels:
            raise ValueError(
                f"Expected {num_levels} anchor sizes, got {len(anchor_sizes)}"
            )

        self.neck = Neck(in_channels, out_channels=neck_n_channels)
        num_anchors = len(anchor_ratios)
        self.heads = nn.ModuleList([
            Head(neck_n_channels, num_anchors, num_classes)
            for _ in range(num_levels)
        ])

        anchor_generator = AnchorGenerator(
            sizes=tuple((size,) for size in anchor_sizes),
            aspect_ratios=tuple(
                tuple(anchor_ratios) for _ in range(num_levels)
            ),
        )
        grid_sizes = [
            (input_size[0] // reduction, input_size[1] // reduction)
            for reduction in reductions
        ]
        strides = [(reduction, reduction) for reduction in reductions]
        anchors = torch.cat(
            anchor_generator.grid_anchors(grid_sizes, strides), dim=0
        ).unsqueeze(0)
        self.register_buffer("anchors", anchors)
        self.register_buffer(
            "anchor_centers", (anchors[..., :2] + anchors[..., 2:]) / 2
        )
        self.register_buffer(
            "anchor_sizes", anchors[..., 2:] - anchors[..., :2]
        )

    def forward(self, x):
        features = self.backbone(x)
        pyramid = self.neck(features)
        cls_outputs = []
        reg_outputs = []

        for head, feature in zip(self.heads, pyramid):
            cls_logits, bbox_preds = head(feature)
            batch_size = x.shape[0]
            cls_outputs.append(
                cls_logits.permute(0, 2, 3, 1)
                .contiguous()
                .view(batch_size, -1, self.num_classes)
            )
            reg_outputs.append(
                bbox_preds.permute(0, 2, 3, 1)
                .contiguous()
                .view(batch_size, -1, 5)
            )

        cls_logits = torch.cat(cls_outputs, dim=1)
        bbox_preds = torch.cat(reg_outputs, dim=1)
        bbox_offsets = bbox_preds[..., :4]
        confidence_logits = bbox_preds[..., 4]

        if self.training:
            return bbox_offsets, confidence_logits, cls_logits

        bboxes = self.decode_bboxes(bbox_offsets)
        return bboxes, confidence_logits.sigmoid(), cls_logits.softmax(dim=-1)

    def decode_bboxes(self, bbox_offsets):
        single_image = bbox_offsets.ndim == 2
        if single_image:
            bbox_offsets = bbox_offsets.unsqueeze(0)

        if bbox_offsets.shape[1] != self.anchors.shape[1]:
            raise ValueError(
                f"Predictions contain {bbox_offsets.shape[1]} anchors, "
                f"but the detector stores {self.anchors.shape[1]}"
            )

        tx, ty, tw, th = bbox_offsets.unbind(dim=-1)
        centers = self.anchor_centers
        sizes = self.anchor_sizes
        center_x = centers[..., 0] + tx.sigmoid() * sizes[..., 0]
        center_y = centers[..., 1] + ty.sigmoid() * sizes[..., 1]
        width = tw.clamp(-10, 10).exp() * sizes[..., 0]
        height = th.clamp(-10, 10).exp() * sizes[..., 1]
        boxes = torch.stack(
            [center_x - width / 2, center_y - height / 2,
             center_x + width / 2, center_y + height / 2],
            dim=-1,
        )
        return boxes[0] if single_image else boxes

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [81]:
from torchvision.ops import box_iou


def TAL_assigner(
    anchors,
    gt_boxes,
    gt_labels,
    num_classes,
    pred_boxes=None,
    pred_cls_logits=None,
    top_k=5,
    alpha=1.0,
    beta=6.0,
):
    """Assign anchors using the Task Alignment metric."""
    device = anchors.device
    num_anchors = anchors.shape[0]
    target_offsets = torch.zeros((num_anchors, 4), device=device)
    target_objectness = torch.zeros(num_anchors, device=device)
    target_cls = torch.zeros((num_anchors, num_classes), device=device)

    if gt_boxes.numel() == 0:
        return target_offsets, target_objectness, target_cls

    gt_xyxy = gt_boxes
    candidate_boxes = anchors if pred_boxes is None else pred_boxes
    ious = box_iou(candidate_boxes, gt_xyxy)
    anchor_centers = (anchors[:, :2] + anchors[:, 2:]) / 2
    if pred_cls_logits is None:
        class_scores = torch.ones(
            (num_anchors, num_classes), device=device, dtype=anchors.dtype
        )
    else:
        class_scores = pred_cls_logits.sigmoid()

    selected = torch.zeros(
        (num_anchors, gt_xyxy.shape[0]), dtype=torch.bool, device=device
    )
    for gt_index, label in enumerate(gt_labels.tolist()):
        inside = (
            (anchor_centers[:, 0] >= gt_xyxy[gt_index, 0])
            & (anchor_centers[:, 0] <= gt_xyxy[gt_index, 2])
            & (anchor_centers[:, 1] >= gt_xyxy[gt_index, 1])
            & (anchor_centers[:, 1] <= gt_xyxy[gt_index, 3])
        )
        alignment = class_scores[:, label].clamp_min(0).pow(alpha)
        alignment = alignment * ious[:, gt_index].clamp_min(0).pow(beta)
        alignment[~inside] = -1
        selected_indices = torch.topk(
            alignment, k=min(top_k, num_anchors)
        ).indices
        valid = alignment[selected_indices] >= 0
        if valid.any():
            selected_indices = selected_indices[valid]
        else:
            selected_indices = ious[:, gt_index].argmax().view(1)
        selected[selected_indices, gt_index] = True

    overlaps = ious.masked_fill(~selected, -1)
    best_iou, best_gt = overlaps.max(dim=1)
    for anchor_index in (best_iou >= 0).nonzero(as_tuple=True)[0].tolist():
        gt_index = best_gt[anchor_index]
        anchor = anchors[anchor_index]
        gt = gt_xyxy[gt_index]
        anchor_center = (anchor[:2] + anchor[2:]) / 2
        anchor_size = (anchor[2:] - anchor[:2]).clamp_min(1e-6)
        gt_center = (gt[:2] + gt[2:]) / 2
        gt_size = (gt[2:] - gt[:2]).clamp_min(1e-6)
        target_offsets[anchor_index, :2] = (gt_center - anchor_center) / anchor_size
        target_offsets[anchor_index, 2:] = torch.log(gt_size / anchor_size)
        target_objectness[anchor_index] = 1
        target_cls[anchor_index, gt_labels[gt_index]] = 1

    return target_offsets, target_objectness, target_cls

### DIoU [1]

Вместо SmoothL1 реализуем Distance Intersection over Union (DIoU).

#### 1. Intersection over Union

Пусть предсказанный и истинный боксы заданы в формате XYXY:

$$B^p=(x^p_1, y^p_1, x^p_2, y^p_2), \qquad B^g=(x^g_1, y^g_1, x^g_2, y^g_2).$$

Координаты пересечения:

$$x^I_1 = \max(x^p_1, x^g_1), \qquad y^I_1 = \max(y^p_1, y^g_1)$$
$$x^I_2 = \min(x^p_2, x^g_2), \qquad y^I_2 = \min(y^p_2, y^g_2)$$

Если $x^I_2 > x^I_1$ и $y^I_2 > y^I_1$, то:

$$I = (x^I_2 - x^I_1)(y^I_2 - y^I_1),$$

иначе $I=0$.

Площади боксов и объединение:

$$A^p=(x^p_2-x^p_1)(y^p_2-y^p_1),$$
$$A^g=(x^g_2-x^g_1)(y^g_2-y^g_1),$$
$$U=A^p+A^g-I,$$
$$IoU=\frac{I}{U}.$$

#### 2. Выпуклая оболочка

Координаты минимального охватывающего бокса:

$$x^c_1 = \min(x^p_1, x^g_1), \qquad y^c_1 = \min(y^p_1, y^g_1)$$
$$x^c_2 = \max(x^p_2, x^g_2), \qquad y^c_2 = \max(y^p_2, y^g_2)$$

Квадрат его диагонали:

$$c^2=(x^c_2-x^c_1)^2+(y^c_2-y^c_1)^2.$$

#### 3. Расстояние между центрами

Центры боксов:

$$C^p=\left(\frac{x^p_1+x^p_2}{2},\frac{y^p_1+y^p_2}{2}\right), \qquad
C^g=\left(\frac{x^g_1+x^g_2}{2},\frac{y^g_1+y^g_2}{2}\right).$$

Квадрат расстояния между центрами:

$$d^2=(C^p_x-C^g_x)^2+(C^p_y-C^g_y)^2.$$

Итоговый DIoU loss:

$$DIoU=1-IoU+\frac{d^2}{c^2}.$$

Для нескольких пар боксов возвращаем среднее значение loss.

In [43]:
from torchvision.ops import distance_box_iou_loss

In [44]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [45]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [46]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 0.9992188215255737


In [47]:
def diou_loss(pred_boxes, gt_boxes):
    """Mean Distance-IoU loss for boxes in xyxy format."""
    if pred_boxes.shape != gt_boxes.shape or pred_boxes.shape[-1] != 4:
        raise ValueError("pred_boxes and gt_boxes must have shape [N, 4]")
    if pred_boxes.numel() == 0:
        return pred_boxes.new_tensor(0.0)

    pred_size = (pred_boxes[:, 2:] - pred_boxes[:, :2]).clamp_min(0)
    gt_size = (gt_boxes[:, 2:] - gt_boxes[:, :2]).clamp_min(0)
    pred_area = pred_size[:, 0] * pred_size[:, 1]
    gt_area = gt_size[:, 0] * gt_size[:, 1]

    intersection_min = torch.maximum(pred_boxes[:, :2], gt_boxes[:, :2])
    intersection_max = torch.minimum(pred_boxes[:, 2:], gt_boxes[:, 2:])
    intersection_size = (intersection_max - intersection_min).clamp_min(0)
    intersection = intersection_size[:, 0] * intersection_size[:, 1]
    union = (pred_area + gt_area - intersection).clamp_min(1e-7)
    iou = intersection / union

    enclosing_min = torch.minimum(pred_boxes[:, :2], gt_boxes[:, :2])
    enclosing_max = torch.maximum(pred_boxes[:, 2:], gt_boxes[:, 2:])
    enclosing_size = enclosing_max - enclosing_min
    diagonal_squared = (enclosing_size ** 2).sum(dim=1).clamp_min(1e-7)

    pred_centers = (pred_boxes[:, :2] + pred_boxes[:, 2:]) / 2
    gt_centers = (gt_boxes[:, :2] + gt_boxes[:, 2:]) / 2
    center_distance_squared = ((pred_centers - gt_centers) ** 2).sum(dim=1)
    return (1 - iou + center_distance_squared / diagonal_squared).mean()

In [48]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

В этой работе использованы аугментации, FPN, decoupled head, TAL и DIoU loss.

### Ответы на вопросы

**1. Какой метод label assignment помогает лучше обучаться модели?**

TAL обычно обучает модель стабильнее, чем assignment только по фиксированному IoU-порогу. Он учитывает одновременно качество классификации, IoU и положение центра anchor внутри GT. Поэтому положительными становятся наиболее подходящие для конкретного объекта предсказания, а неоднозначные anchors не получают положительный label только из-за одного порога. Это уменьшает шум в target assignment и помогает классификации и локализации обучаться согласованно.

**2. Какое улучшение должно дать наибольший вклад в качество?**

Наибольший вклад обычно даёт FPN. Объекты на изображениях имеют разные размеры, а FPN объединяет признаки нескольких пространственных масштабов. Верхние уровни содержат сильную семантику, а нижние сохраняют больше пространственных деталей. Дополнительный вклад могут дать разморозка последних блоков backbone и decoupled head, но их влияние нужно подтверждать одинаковым экспериментальным протоколом.

**3. Какое улучшение может не изменить метрику?**

Наименьшее изменение может дать шумовая или цветовая аугментация, если исходные изображения уже имеют похожее разнообразие освещения и качества. Также эффект может быть незаметен при коротком обучении, слишком высоком learning rate или если ошибка модели в основном связана с assignment и декодированием боксов. Поэтому сравнивать улучшения нужно при одинаковых seed, числе эпох, optimizer и validation split.

Фактические значения mAP необходимо получить запуском обучения и `validate`: без этого нельзя честно утверждать, какой вариант превысил пороги 0.05, 0.1 или 0.2.

In [82]:
from torch.utils.data import DataLoader
from torchvision.ops import nms


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 1

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

if "model" in globals():
    del model
if "optimizer" in globals():
    del optimizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = Detector(
    backbone_model_name="efficientnet_b0",
    neck_n_channels=128,
    num_classes=4,
    anchor_sizes=(32, 64, 128),
    anchor_ratios=(0.5, 1.0, 2.0),
    input_size=(image_size, image_size),
    unfreeze_last=0,
).to(device)

optimizer = torch.optim.AdamW(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

bbox_loss = nn.SmoothL1Loss()
objectness_loss = nn.BCEWithLogitsLoss()
classification_loss = nn.BCEWithLogitsLoss()


def filter_predictions_xyxy(
    predictions,
    score_threshold=0.1,
    nms_threshold=0.5,
    max_boxes_per_cls=100,
):
    boxes, confidence, class_probs = predictions
    result = []
    for image_boxes, image_confidence, image_probs in zip(
        boxes, confidence, class_probs
    ):
        image_result = {"boxes": [], "labels": [], "scores": []}
        class_scores = image_confidence[:, None] * image_probs
        for class_index in range(class_probs.shape[-1]):
            scores = class_scores[:, class_index]
            keep = scores > score_threshold
            if not keep.any():
                continue
            selected_boxes = image_boxes[keep]
            selected_scores = scores[keep]
            if selected_scores.numel() > max_boxes_per_cls:
                order = torch.argsort(selected_scores, descending=True)[:max_boxes_per_cls]
                selected_boxes = selected_boxes[order]
                selected_scores = selected_scores[order]
            keep_indices = nms(selected_boxes, selected_scores, nms_threshold)
            image_result["boxes"].append(selected_boxes[keep_indices])
            image_result["labels"].append(
                torch.full(
                    (len(keep_indices),),
                    class_index,
                    dtype=torch.int64,
                    device=selected_boxes.device,
                )
            )
            image_result["scores"].append(selected_scores[keep_indices])

        for key in image_result:
            if image_result[key]:
                image_result[key] = torch.cat(image_result[key]).cpu()
            else:
                dtype = torch.int64 if key == "labels" else torch.float32
                shape = (0,) if key != "boxes" else (0, 4)
                image_result[key] = torch.empty(shape, dtype=dtype)
        result.append(image_result)
    return result


def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, targets in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        bbox_offsets, confidence_logits, cls_logits = model(images)
        anchors = model.anchors[0]
        optimizer.zero_grad()
        batch_loss = torch.zeros((), device=device)

        for image_index, target in enumerate(targets):
            gt_boxes = target["boxes"].to(device)
            gt_labels = target["labels"].to(device)
            predicted_boxes = model.decode_bboxes(bbox_offsets[image_index])
            assigned_offsets, assigned_objectness, assigned_classes = TAL_assigner(
                anchors,
                gt_boxes,
                gt_labels,
                model.num_classes,
                pred_boxes=predicted_boxes.detach(),
                pred_cls_logits=cls_logits[image_index].detach(),
            )
            valid = assigned_objectness >= 0
            positive = assigned_objectness == 1
            image_loss = objectness_loss(
                confidence_logits[image_index][valid],
                assigned_objectness[valid],
            )
            if positive.any():
                image_loss = image_loss + bbox_loss(
                    bbox_offsets[image_index][positive],
                    assigned_offsets[positive],
                )
                image_loss = image_loss + classification_loss(
                    cls_logits[image_index][positive],
                    assigned_classes[positive],
                )
            batch_loss = batch_loss + image_loss

        batch_loss = batch_loss / len(targets)
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        total_loss += batch_loss.item()
    return total_loss / max(len(dataloader), 1)


# Запуск эксперимента:
# for epoch in range(10):
#     train_loss = train_one_epoch(model, train_loader, optimizer, device)
#     map_score = validate(
#         model,
#         validation_loader,
#         filter_predictions_xyxy,
#         box_format="xyxy",
#         device=device,
#     )
#     print(f"Epoch {epoch + 1}: loss={train_loss:.4f}, mAP={map_score:.4f}")

In [84]:
from torch.utils.data import DataLoader, Subset

smoke_loader = DataLoader(
    Subset(train_dataset, [0]),
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
)
smoke_loss = train_one_epoch(model, smoke_loader, optimizer, device)
print(f"smoke train loss: {smoke_loss:.4f}")

Training:   0%|          | 0/1 [00:00<?, ?it/s]

smoke train loss: 1.6216


Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [50]:
!pip install torchmetrics

In [51]:
from torchmetrics.detection import MeanAveragePrecision


@torch.no_grad()
def validate(
    model,
    dataloader,
    filter_predictions_func,
    box_format="xyxy",
    device="cpu",
    score_threshold=0.1,
    nms_threshold=0.5,
    **kwargs,
):
    """Evaluate a detector and return mAP at IoU 0.5:0.95."""
    model.eval()
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = model(images)
        predicts = filter_predictions_func(
            outputs, score_threshold, nms_threshold, **kwargs
        )
        metric.update(predicts, targets)
    return metric.compute()["map"].item()

In [76]:
for epoch in range(10):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    map_score = validate(
        model,
        validation_loader,
        filter_predictions_xyxy,
        box_format="xyxy",
        device=device,
    )
    print(f"Epoch {epoch + 1}: loss={train_loss:.4f}, mAP={map_score:.4f}")

Training:   0%|          | 0/462 [00:00<?, ?it/s]

Running validation:   0%|          | 0/71 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


Epoch 1: loss=0.7733, mAP=0.0007


Training:   0%|          | 0/462 [00:00<?, ?it/s]

Running validation:   0%|          | 0/71 [00:00<?, ?it/s]

... (21 outputs left out)

Epoch 9: loss=0.0606, mAP=0.0000


Training:   0%|          | 0/462 [00:00<?, ?it/s]

Running validation:   0%|          | 0/71 [00:00<?, ?it/s]

Epoch 10: loss=0.0574, mAP=0.0000
